In [1]:
# Install required packages (run once)
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow", "matplotlib", "numpy", "torch", "torchvision"])

0

## 1. Import libraries and set dataset path
We import the required libraries and specify the dataset path.

In [2]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

dataset_dir = 'DB/DB1/CUB_200_2011/'

## 2. Load image list and labels
We read images.txt and image_class_labels.txt to build the list of image paths and their labels.

In [3]:
# Read image paths
images_txt = os.path.join(dataset_dir, 'images.txt')
with open(images_txt) as f:
    image_list = [line.strip().split(' ')[1] for line in f]

# Read image labels
labels_txt = os.path.join(dataset_dir, 'image_class_labels.txt')
with open(labels_txt) as f:
    labels = [int(line.strip().split(' ')[1]) for line in f]

print('Number of images:', len(image_list))
print('Number of labels:', len(labels))

Number of images: 11788
Number of labels: 11788


## 3. Load image attributes and concept labels

In [5]:
# Read attribute names
attr_txt = os.path.join('DB', 'DB1', 'attributes.txt')
with open(attr_txt) as f:
    attr_names = [line.strip().split(' ', 1)[1] for line in f]

print('Number of attributes:', len(attr_names))

# Read image attribute labels
attr_label_txt = os.path.join(dataset_dir, 'attributes', 'image_attribute_labels.txt')
image_attr = {}
with open(attr_label_txt) as f:
    for line in f:
        img_id, attr_id, is_present, *_ = line.strip().split()
        img_id = int(img_id)
        attr_id = int(attr_id) - 1
        is_present = int(is_present)
        image_attr.setdefault(img_id, {})[attr_id] = is_present

print('Images with attributes:', len(image_attr))

Number of attributes: 312
Images with attributes: 11788


## 4. Build attribute vectors

Build aligned attribute vectors for each image in image_list

In [6]:
# Build aligned attribute vectors for each image in image_list
num_attrs = len(attr_names)
attr_vectors = []
for i in range(1, len(image_list) + 1):
    vec = np.zeros(num_attrs, dtype=np.int8)
    for attr_id, is_present in image_attr.get(i, {}).items():
        vec[attr_id] = is_present
    attr_vectors.append(vec)

print('Attribute vectors:', len(attr_vectors))

# Show a small sample
sample_id = 1
sample_attrs = [attr_names[i] for i, v in image_attr.get(sample_id, {}).items() if v == 1]
print('Sample image id:', sample_id)
print('Sample attributes (first 5):', sample_attrs[:5])

Attribute vectors: 11788
Sample image id: 1
Sample attributes (first 5): ['has_bill_shape::hooked_seabird', 'has_head_pattern::masked', 'has_throat_color::buff', 'has_eye_color::brown', 'has_bill_length::longer_than_head']


## 5. Split data into train and test
We use train_test_split.txt to separate train and test data.

In [7]:
# Read split file
split_txt = os.path.join(dataset_dir, 'train_test_split.txt')
with open(split_txt) as f:
    is_train = [int(line.strip().split(' ')[1]) for line in f]

# Build train and test lists
train_images      = [img for img, t in zip(image_list,   is_train) if t == 1]
train_labels      = [lbl for lbl, t in zip(labels,       is_train) if t == 1]
train_attr_vecs   = [vec for vec, t in zip(attr_vectors, is_train) if t == 1]

test_images       = [img for img, t in zip(image_list,   is_train) if t == 0]
test_labels       = [lbl for lbl, t in zip(labels,       is_train) if t == 0]
test_attr_vecs    = [vec for vec, t in zip(attr_vectors, is_train) if t == 0]

print('Train samples:', len(train_images))
print('Test samples:', len(test_images))

Train samples: 5994
Test samples: 5794


## 6. Define Level-1 concept groups
We group the 312 attributes into 7 coarse body-part concepts for Level-1 supervision.

In [8]:
import numpy as np

# 7 Level-1 concept groups (0-indexed attribute indices)
CONCEPT_GROUPS = {
    'bill':    list(range(0, 9))   + list(range(149, 152)) + list(range(278, 293)),  # shape+length+color
    'wing':    list(range(9, 24))  + list(range(212, 217)) + list(range(308, 312)),  # color+shape+pattern
    'body':    list(range(24, 73)) + list(range(105, 120)) + list(range(197, 212)) + list(range(236, 248)),
    'head':    list(range(94, 105))+ list(range(120, 149)) + list(range(152, 198)) + list(range(293, 312)),
    'tail':    list(range(73, 94)) + list(range(167, 182)) + list(range(240, 244)),
    'legs':    list(range(263, 278)),
    'overall': list(range(217, 240)) + list(range(248, 263)),
}
CONCEPT_NAMES = list(CONCEPT_GROUPS.keys())
NUM_L1 = len(CONCEPT_NAMES)  # 7
NUM_L2 = len(attr_names)     # 312

def build_l1_labels(attr_vecs):
    """For each image, compute binary Level-1 labels (1 if any attr in group is 1)."""
    result = []
    for vec in attr_vecs:
        row = [int(any(vec[i] for i in idxs)) for idxs in CONCEPT_GROUPS.values()]
        result.append(row)
    return np.array(result, dtype=np.float32)

train_l1 = build_l1_labels(train_attr_vecs)
test_l1  = build_l1_labels(test_attr_vecs)
train_l2 = np.array(train_attr_vecs, dtype=np.float32)
test_l2  = np.array(test_attr_vecs,  dtype=np.float32)

print('L1 shape (train):', train_l1.shape)   # (N_train, 7)
print('L2 shape (train):', train_l2.shape)   # (N_train, 312)
print('Concept groups:', CONCEPT_NAMES)

L1 shape (train): (5994, 7)
L2 shape (train): (5994, 312)
Concept groups: ['bill', 'wing', 'body', 'head', 'tail', 'legs', 'overall']


## 7. Load Segmentation Masks
Load the binary segmentation masks from DB1-Mask. Each mask marks which pixels belong to the bird (1) vs background (0).

In [ ]:
# Segmentation mask root directory
mask_root = os.path.join(_project_root, 'DB', 'DB1-Mask', 'segmentations')

# Transform for masks: resize to 224x224, convert to tensor (values 0 or 1)
mask_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),   # 1 = bird pixel, 0 = background pixel
])

# Verify masks exist and count available files
mask_count = sum(
    1 for root, _, files in os.walk(mask_root)
    for f in files if f.endswith('.png')
)
print(f'Mask root: {mask_root}')
print(f'Total mask files found: {mask_count}')

## 8. PyTorch Dataset and DataLoader
Wraps images, labels, attributes, and masks into a unified Dataset for training.

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE   = 224
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNet stats
])

class BirdDataset(Dataset):
    def __init__(self, image_paths, labels, l1_labels, l2_labels,
                 img_root, mask_root, transform=None, mask_transform=None):
        self.paths          = image_paths
        self.labels         = labels        # class label (1-indexed → converted to 0-indexed)
        self.l1             = l1_labels     # (N, 7)   coarse concepts
        self.l2             = l2_labels     # (N, 312) fine attributes
        self.img_root       = img_root
        self.mask_root      = mask_root
        self.transform      = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        rel_path = self.paths[idx]   # e.g. '001.Black_footed_Albatross/img_001.jpg'

        # --- image ---
        img = Image.open(os.path.join(self.img_root, 'images', rel_path)).convert('RGB')
        if self.transform:
            img = self.transform(img)

        # --- mask (same relative path, .png extension, under mask_root) ---
        mask_path = os.path.join(self.mask_root,
                                 os.path.splitext(rel_path)[0] + '.png')
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('L')   # grayscale
            if self.mask_transform:
                mask = self.mask_transform(mask)         # (1, 224, 224)
        else:
            mask = torch.zeros(1, IMG_SIZE, IMG_SIZE)    # fallback: all background

        # --- labels ---
        label = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        l1    = torch.tensor(self.l1[idx], dtype=torch.float32)
        l2    = torch.tensor(self.l2[idx], dtype=torch.float32)

        return img, label, l1, l2, mask


train_dataset = BirdDataset(train_images, train_labels, train_l1, train_l2,
                            dataset_dir, mask_root, transform, mask_transform)
test_dataset  = BirdDataset(test_images,  test_labels,  test_l1,  test_l2,
                            dataset_dir, mask_root, transform, mask_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches: ', len(test_loader))

# Verify one batch
imgs, lbls, l1s, l2s, masks = next(iter(train_loader))
print('Image batch:', imgs.shape)    # (32, 3, 224, 224)
print('Label batch:', lbls.shape)    # (32,)
print('L1 batch:   ', l1s.shape)     # (32, 7)
print('L2 batch:   ', l2s.shape)     # (32, 312)
print('Mask batch: ', masks.shape)   # (32, 1, 224, 224)

Train batches: 188
Test batches:  182
Image batch: torch.Size([32, 3, 224, 224])
Label batch: torch.Size([32])
L1 batch:    torch.Size([32, 7])
L2 batch:    torch.Size([32, 312])


# Data Pipeline Roadmap

---

## Section 1 — Install Packages

**Input:** Nothing — runs once at the beginning.

**Process:** Installs all required packages via `pip`:
- `pillow` — for opening and processing images
- `numpy` — for numerical array operations
- `torch` & `torchvision` — the core deep learning framework
- `matplotlib` — for plotting and displaying images

**Output:** Python environment ready; all imports available.

---

## Section 2 — Set Dataset Path

**Input:** Nothing — path is defined manually.

**Process:** Sets the `dataset_dir` variable pointing to the root folder of the CUB-200-2011 dataset. This variable is reused across all subsequent sections so the path never needs to be repeated.

**Output:**
```
dataset_dir = 'DB/DB1/CUB_200_2011/'
```

---

## Section 3 — Load Images, Labels & Attributes

**Input:** Four plain-text files from disk:
- `images.txt` — index and file path of each image
- `image_class_labels.txt` — bird species class for each image
- `attributes.txt` — names of the 312 attributes
- `image_attribute_labels.txt` — attribute values (0/1) per image

**Process:** Reads each file line by line and splits the values. Attribute data (stored as `img_id, attr_id, is_present` triplets) is loaded into a nested dictionary. The resulting structure is **sparse** — only observed values are stored; not all 312 slots are written for every image.

**Output:**
```
image_list  → list of 11,788 strings  e.g. '001.Black_footed_Albatross/img_001.jpg'
labels      → list of 11,788 integers (1–200) — class per image
attr_names  → list of 312 strings     e.g. 'has_bill_shape::curved', 'has_wing_color::blue'
image_attr  → { 1: {2: 1, 7: 1, 45: 0, ...},
                2: {0: 0, 11: 1, ...},
                ... }   ← sparse dictionary
```

---

## Section 4 — Build Attribute Vectors (Sparse → Dense)

**Input:** `image_attr` (sparse dictionary from Section 3) and `attr_names` to know the fixed length (312).

**Process:** Converting sparse dictionary to a fixed-size dense array.

```
image_attr[1] = {2: 1, 7: 1, 45: 0}     ← sparse (incomplete, unordered)
        │
        ▼
[0, 0, 1, 0, 0, 0, 0, 1, 0, ..., 0]     ← dense (complete, ordered, length 312)
```

**Output:**
```
attr_vectors → list of 11,788 numpy arrays, each of shape (312,) with values 0 or 1
```

---

## Section 5 — Train / Test Split

**Input:** `image_list`, `labels`, `attr_vectors` (all 11,788 entries) + `train_test_split.txt`.

**Process:** Reads the split file — each line contains a flag: `1` = train, `0` = test. Uses list comprehension to filter all three lists simultaneously. This split was predefined by the CUB dataset authors and is the standard benchmark split, ensuring results are reproducible and comparable across research papers.

**Output:**
```
train_images    → 5,994 image paths
train_labels    → 5,994 class labels
train_attr_vecs → 5,994 arrays of shape (312,)

test_images     → 5,794 image paths
test_labels     → 5,794 class labels
test_attr_vecs  → 5,794 arrays of shape (312,)
```

---

## Section 6 — Build L1 and L2 Labels

**Input:** `train_attr_vecs` and `test_attr_vecs` — arrays of shape (N, 312).

**Process:** Creates two levels of supervision labels for the Multi-Level CBM model:

- **L2 (Fine-grained):** Keeps the full 312-element array as-is. Each slot answers a specific question like "Is the wing color blue?"

- **L1 (Coarse):** Groups the 312 attributes into 7 high-level concept groups. For each group, checks if **at least one** attribute in that group equals 1. If yes → that group = 1.

```
312-element array:  [0, 0, 1, 0, ..., 1, ..., 0, 0, 1, ...]
                             ↑          ↑           ↑
                     bill_shape=1  wing_color=1  tail_shape=1
                          │              │             │
                          ▼              ▼             ▼
7-element array:   [ bill=1,        wing=1,       tail=1,  body=0, head=0, legs=0, overall=0 ]
```

**Output:**
```
train_l2 → numpy array of shape (5994, 312)  — fine-grained attribute labels
train_l1 → numpy array of shape (5994,   7)  — coarse concept labels
test_l2  → numpy array of shape (5794, 312)
test_l1  → numpy array of shape (5794,   7)
```

### The 7 L1 Concept Groups

| Group | Sub-categories |
|-------|---------------|
| **bill** | bill_shape (9), bill_length (3), bill_color (15) |
| **wing** | wing_color (15), wing_shape (5), wing_pattern (4) |
| **body** | upperparts_color, underparts_color, back_color, breast_color, belly_color + patterns |
| **head** | head_pattern (11), throat_color (15), eye_color (14), forehead_color (15), nape_color (15), crown_color (15) |
| **tail** | tail_shape (6), upper_tail_color (15), under_tail_color (15), tail_pattern (4) |
| **legs** | leg_color (15) |
| **overall** | size (5), shape (14), back_pattern (4), primary_color (15) |

> The original CUB dataset has 28 categories summing to 312 attributes. These 7 groups are our custom design for the L1 level. Some attributes intentionally appear in more than one group.

---

## Section 7 — Load Segmentation Masks

**Input:** PNG mask files from `DB/DB1-Mask/segmentations/` — one mask per image, same filename as the image but with `.png` extension.

**Process:** Defines `mask_root` pointing to the segmentation folder and `mask_transform` to resize masks to 224×224 and convert them to tensors. Verifies the mask files exist by counting them. Each mask is a binary grayscale image where:
- **1 (white)** = bird pixel
- **0 (black)** = background pixel

These masks are not used during training. They are reserved for **Section 5 (XAI)** to evaluate whether the model's attention maps align with the actual bird region — a metric known as *Faithfulness* or *Localization Score*.

**Output:**
```
mask_root      → path string to DB/DB1-Mask/segmentations/
mask_transform → torchvision transform (Resize + ToTensor)
```

---

## Section 8 — PyTorch Dataset & DataLoader

**Input:** `train_images`, `train_labels`, `train_l1`, `train_l2`, `mask_root`, `mask_transform` + path to the images folder.

**Process:** Defines the `BirdDataset` class which on each `__getitem__` call:
1. Opens the image from disk using PIL and applies `transform` (resize, normalize)
2. Loads the corresponding mask (falls back to a zero tensor if the mask file is missing)
3. Returns the image, label, l1, l2, and mask together as tensors

Then wraps the dataset in a `DataLoader` that delivers shuffled **batches of 32** samples during training.

**Output:**
```
train_loader and test_loader — each batch contains:

  imgs  → tensor (32, 3, 224, 224)  ← 32 RGB images at 224×224
  lbls  → tensor (32,)              ← class index per image (0-indexed)
  l1s   → tensor (32, 7)            ← coarse concept labels
  l2s   → tensor (32, 312)          ← fine-grained attribute labels
  masks → tensor (32, 1, 224, 224)  ← binary segmentation masks
```